In [1]:
import pandas as pd
from pyproj import Proj, transform
from geopy.geocoders import Nominatim
import pyproj
import time
import re
import numpy as np
from tqdm import tqdm
from geopy.exc import GeocoderTimedOut
import math

In [2]:
#Unidae comercial
und = pd.read_csv("unidade_comercial.csv", encoding='latin1', delimiter='\t')
und.columns

und['Número'] = np.nan
und['Latitude'] = np.nan
und['Longitude'] = np.nan


C:\Users\Projeto 7\AppData\Local\Temp\ipykernel_14268\939843541.py:2: DtypeWarning: Columns (23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  und = pd.read_csv("unidade_comercial.csv", encoding='latin1', delimiter='\t')


In [3]:
pd.set_option('display.max_rows', None)
und['Serviço Executado'].value_counts()

Serviço Executado
800 - RECADASTRAMENTO                                          4981
801 - RECADASTRAMENTO EXECUTADO                                4910
813 - ALTERAR DADOS DA LIGACAO - RECADASTRAMENTO               3698
24 - AC - Emissao 2 Via de Fatura                              2925
810 - ALTERACAO DADOS CADASTRAIS - RECADASTRAMENTO             2675
802 - RECADASTRAMENTO 2º VISITA                                1089
3101 - LA - Relig/Corte Cav.                                    769
812 - ALTERAR DADOS DE FONTE PROPRIA - RECADASTRAMENTO          647
3100 - LA - Corte Cav. Falta Pagamento                          640
5002 - AC - Alterar Cliente Unidade Comercial                   562
2222 - CANCELAR PARCELA LIXO - BOLETO                           537
811 - ALTERACAO CATEGORIA/ECONOMIA - RECADASTRAMENTO            329
814 - ALTERAR ENDERECO PRINCIPAL - RECADASTRAMENTO              327
2020 - LA -  Conserto Ramal                                     254
5028 - CAD -  Alterar Dados En

In [4]:
pd.set_option('display.max_rows', 10)
und = und[und['Serviço Executado'].str.startswith('4000') | und['Serviço Executado'].isnull()]
und['Categoria de serviço'] = 'Falta de água'


In [5]:
und

,Data de Solicitação,Número da OS,Código Logradouro,Endereço,Ponto de Referência,Bairro,Município,Matrícula,Situação Ligação Água,Serviço Solicitado,...,Data Final da Suspensão por Controle,Motivo da Suspensão por Controle,Parecer da Suspensão por Controle,Período de Suspensão por Controle (dias - horas:min:seg),Data Final da Liberação,Parecer da Liberação,Número,Latitude,Longitude,Categoria de serviço
6,02/01/2023 15:55:00,115757,287,Rua 182 - OTTO VOLLES 291 - Rio Hern -,NaN,Rio Hern,NaN,1366026-8,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
65,06/01/2023 09:54:00,115821,287,Rua 182 - OTTO VOLLES 253 - Rio Hern -,NaN,Rio Hern,NaN,1353886-1,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
67,06/01/2023 10:12:00,115823,48,R. GUAIBA 64 - Rio Hern -,NaN,Rio Hern,NaN,753284-9,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
347,11/01/2023 13:13:00,116115,49,R. ERICH FROEHNER 3978 - Schroeder I . - Sala...,NaN,Schroeder I .,NaN,1369009-4,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
1174,09/02/2023 11:06:00,116979,346,Rua ESTRADA DUAS MAMAS 4962 - Duas Mamas - CX ...,NaN,Duas Mamas,NaN,1368366-7,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13484,25/05/2023 15:46:00,159880,307,Rua 573- VERONICA WELTER 156 - Schroeder I -,NaN,Schroeder I,NaN,1365896-4,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
17282,19/06/2023 15:10:00,178079,10,R. Santa. Catarina 15 - Braco do Sul -,NaN,Braco do Sul,NaN,1369024-8,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
20006,30/06/2023 10:13:00,181364,3,R. Duque de Caxias 381 - Centro Norte -,NaN,Centro Norte,NaN,801221-0,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água
29384,08/09/2023 14:21:00,192215,7,R. PRES. COSTA E SILVA 1291 - Rio Hern -,NaN,Rio Hern,NaN,1354221-4,NaN,4000 - LA - Verificar Falta de Agua,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Falta de água


In [6]:
und = und[['Matrícula','Categoria de serviço','Data de Solicitação','Endereço','Bairro','Número','Latitude','Longitude']]
und.to_csv("Unidade comercial tratado.csv",sep=';',encoding = 'latin1')
und

,Matrícula,Categoria de serviço,Data de Solicitação,Endereço,Bairro,Número,Latitude,Longitude
6,1366026-8,Falta de água,02/01/2023 15:55:00,Rua 182 - OTTO VOLLES 291 - Rio Hern -,Rio Hern,NaN,NaN,NaN
65,1353886-1,Falta de água,06/01/2023 09:54:00,Rua 182 - OTTO VOLLES 253 - Rio Hern -,Rio Hern,NaN,NaN,NaN
67,753284-9,Falta de água,06/01/2023 10:12:00,R. GUAIBA 64 - Rio Hern -,Rio Hern,NaN,NaN,NaN
347,1369009-4,Falta de água,11/01/2023 13:13:00,R. ERICH FROEHNER 3978 - Schroeder I . - Sala...,Schroeder I .,NaN,NaN,NaN
1174,1368366-7,Falta de água,09/02/2023 11:06:00,Rua ESTRADA DUAS MAMAS 4962 - Duas Mamas - CX ...,Duas Mamas,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
13484,1365896-4,Falta de água,25/05/2023 15:46:00,Rua 573- VERONICA WELTER 156 - Schroeder I -,Schroeder I,NaN,NaN,NaN
17282,1369024-8,Falta de água,19/06/2023 15:10:00,R. Santa. Catarina 15 - Braco do Sul -,Braco do Sul,NaN,NaN,NaN
20006,801221-0,Falta de água,30/06/2023 10:13:00,R. Duque de Caxias 381 - Centro Norte -,Centro Norte,NaN,NaN,NaN
29384,1354221-4,Falta de água,08/09/2023 14:21:00,R. PRES. COSTA E SILVA 1291 - Rio Hern -,Rio Hern,NaN,NaN,NaN
